##SGG Pipeline — Validation & Verification
Accuracy + Latency — Runs WITHOUT the trained LLM classifier




##What this notebook validates:
Environment CNN — ResNet18 fine-tuned on Places365
Scene Graph Generator — YOLO-World SGG
Metrics measured:
* ✅ Environment classification accuracy (top-1 & top-3)
* ✅ Per-class precision, recall, F1
* ✅ Confusion matrix
* ✅ YOLO-World detection rate (objects found per frame)
* ✅ SGG triplet quality (nodes & edges generated)
* ✅ Latency per stage (CNN ms/frame, YOLO ms/frame, full pipeline ms/video)
* ✅ Full end-to-end pipeline smoke test
* ✅ Summary report saved to validation_report.json

In [69]:
!pip install ultralytics

In [70]:
import torch



# Path to your trained ResNet18 .pth file
ENV_MODEL_PATH   = "/content/drive/MyDrive/Other/places365_environment_model_new.pth"

# Class names — MUST match the order used during training
CLASSES          = ["classroom", "home", "office"]

# Option A: folder of labelled test videos (leave None to skip)
TEST_DIR         = "/content/drive/MyDrive/Colab Notebooks/Data"          # e.g. "/content/drive/MyDrive/Data/test_videos"

# Option B: single video quick check (leave None to skip)
#SINGLE_VIDEO     ="/content/drive/MyDrive/Colab Notebooks/Data/living_room/3769952-hd_1920_1080_25fps.mp4"          # e.g. "office-trimmed.mp4"
#SINGLE_LABEL     = "home"         # e.g. "office"

# YOLO-World weights (auto-downloaded by ultralytics if not present)
YOLO_WEIGHTS     = "yolov8s-world.pt"

# Set True to skip YOLO/SGG tests (CNN only)
NO_SGG           = False

# Output report path
REPORT_PATH      = "validation_report.json"

# Frames sampled per video
NUM_SAMPLE_FRAMES = 5

# SGG object vocabulary
DEFAULT_SGG_VOCAB = [
    "person", "gun", "knife",
    "fire", "flames", "smoke", "fireplace", "burning",
    "person falling", "chair", "desk", "laptop", "projector",
]

print("✅ Configuration set.")
print(f"   Model  : {ENV_MODEL_PATH}")
print(f"   Classes: {CLASSES}")
print(f"   Device : {'GPU' if torch.cuda.is_available() else 'CPU'}")

✅ Configuration set.
   Model  : /content/drive/MyDrive/Other/places365_environment_model_new.pth
   Classes: ['classroom', 'home', 'office']
   Device : CPU


In [71]:
"""
=============================================================================
  SGG PIPELINE — VALIDATION & VERIFICATION SCRIPT
  (Accuracy + Latency) —
=============================================================================
"""

import os
import sys
import cv2
import time
import json
import warnings
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torchvision import transforms
import torchvision.models as models
from PIL import Image

try:
    from ultralytics import YOLOWorld
    YOLO_AVAILABLE = True
    print("✅ ultralytics / YOLO-World available")
except ImportError:
    YOLO_AVAILABLE = False
    print("⚠️  ultralytics not installed — SGG tests will be skipped.")

print("✅ All imports successful.")

✅ ultralytics / YOLO-World available
✅ All imports successful.


In [72]:
# ── Colour helpers (terminal pretty-print) ────────────────────────────────────
class C:
    HEADER  = "\033[95m"
    BLUE    = "\033[94m"
    CYAN    = "\033[96m"
    GREEN   = "\033[92m"
    YELLOW  = "\033[93m"
    RED     = "\033[91m"
    BOLD    = "\033[1m"
    END     = "\033[0m"

def banner(title):
    line = "═" * 60
    print(f"\n{C.BOLD}{C.BLUE}{line}{C.END}")
    print(f"{C.BOLD}{C.BLUE}  {title}{C.END}")
    print(f"{C.BOLD}{C.BLUE}{line}{C.END}")

def ok(msg):   print(f"  {C.GREEN}✔  {msg}{C.END}")
def warn(msg): print(f"  {C.YELLOW}⚠  {msg}{C.END}")
def fail(msg): print(f"  {C.RED}✘  {msg}{C.END}")
def info(msg): print(f"  {C.CYAN}ℹ  {msg}{C.END}")

print("✅ Helper functions defined.")

✅ Helper functions defined.


In [73]:
def load_env_model(model_path: str, num_classes: int = 3) -> tuple:
    """Load the fine-tuned ResNet18 environment classifier."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet18(num_classes=365)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model, device


def load_sgg_model(weights: str = YOLO_WEIGHTS, vocab: list = None):
    """Load YOLO-World and configure vocabulary."""
    if not YOLO_AVAILABLE:
        return None
    model = YOLOWorld(weights)
    if vocab:
        model.set_classes(vocab)
    return model


# ── Preprocessing transform (same as training notebook) ───────────────────────
PREPROCESS = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

print("✅ Model loader functions defined.")

✅ Model loader functions defined.


In [74]:
def calculate_iou(boxA, boxB):
    """Compute Intersection over Union for two bounding boxes."""
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return interArea / float(areaA + areaB - interArea + 1e-5)


def sample_frames(video_path: str, n: int = NUM_SAMPLE_FRAMES):
    """
    Yield (frame_index, bgr_frame) for n evenly-spaced frames.
    Also returns fps and total_frames as metadata.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_val = cap.get(cv2.CAP_PROP_FPS) or 30.0
    indices = set(np.linspace(0, total - 1, n, dtype=int).tolist())

    frames = []
    for i in range(total):
        ret, frame = cap.read()
        if not ret:
            break
        if i in indices:
            frames.append((i, frame))

    cap.release()
    return frames, fps_val, total


class LatencyTimer:
    """Accumulates timing measurements and computes statistics."""

    def __init__(self, name: str):
        self.name = name
        self.times: list = []

    def measure(self, fn, *args, **kwargs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        self.times.append((time.perf_counter() - t0) * 1000)  # ms
        return result

    def stats(self) -> dict:
        if not self.times:
            return {"n": 0, "mean_ms": 0, "min_ms": 0, "max_ms": 0, "p95_ms": 0}
        arr = np.array(self.times)
        return {
            "n":       len(arr),
            "mean_ms": round(float(np.mean(arr)),   2),
            "min_ms":  round(float(np.min(arr)),    2),
            "max_ms":  round(float(np.max(arr)),    2),
            "p95_ms":  round(float(np.percentile(arr, 95)), 2),
        }

    def report(self):
        s = self.stats()
        info(f"{self.name:30s}  "
             f"mean={s['mean_ms']:7.1f}ms  "
             f"min={s['min_ms']:7.1f}ms  "
             f"max={s['max_ms']:7.1f}ms  "
             f"p95={s['p95_ms']:7.1f}ms  "
             f"(n={s['n']})")

print("✅ Utility functions defined.")

✅ Utility functions defined.


In [75]:
# ── Stage A: CNN Environment Classifier Accuracy & Latency ────────────────────

def validate_env_model(env_model, device, test_videos: list, classes: list):
    """
    Run the CNN on every sampled frame of every test video.
    test_videos: list of (video_path, true_label_str)
    """
    banner("STAGE A — Environment CNN Accuracy & Latency")

    if not test_videos:
        warn("No labelled test videos provided. Skipping accuracy test.")
        return {}

    timer         = LatencyTimer("CNN inference / frame")
    y_true        = []
    y_pred        = []
    y_conf        = []
    video_results = []

    for video_path, true_label in test_videos:
        if true_label not in classes:
            warn(f"Label '{true_label}' not in class list — skipping {video_path}")
            continue

        true_idx = classes.index(true_label)

        try:
            frames, _, _ = sample_frames(video_path)
        except IOError as e:
            fail(str(e))
            continue

        accumulated = torch.zeros(len(classes)).to(device)
        per_frame_preds = []

        for _, bgr in frames:
            rgb    = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            pil    = Image.fromarray(rgb)
            tensor = PREPROCESS(pil).unsqueeze(0).to(device)

            def _infer():
                with torch.no_grad():
                    out   = env_model(tensor)
                    probs = torch.nn.functional.softmax(out[0], dim=0)
                return probs

            probs = timer.measure(_infer)
            accumulated += probs
            per_frame_preds.append(int(probs.argmax().item()))

        avg_probs = accumulated / len(frames)
        conf, pred_idx = torch.max(avg_probs, 0)

        y_true.append(true_idx)
        y_pred.append(pred_idx.item())
        y_conf.append(conf.item())

        correct      = (pred_idx.item() == true_idx)
        symbol       = "✔" if correct else "✘"
        status_color = C.GREEN if correct else C.RED
        print(f"  {status_color}{symbol}{C.END}  "
              f"{Path(video_path).name:35s}  "
              f"true={true_label:12s}  "
              f"pred={classes[pred_idx.item()]:12s}  "
              f"conf={conf.item():.3f}")

        video_results.append({
            "video":      video_path,
            "true_label": true_label,
            "pred_label": classes[pred_idx.item()],
            "confidence": round(conf.item(), 4),
            "correct":    correct,
        })

    # Overall metrics
    n_total   = len(y_true)
    n_correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    accuracy  = n_correct / n_total if n_total else 0.0

    banner("ENV CNN — Results Summary")
    ok(f"Top-1 Accuracy : {n_correct}/{n_total} = {accuracy*100:.1f}%")
    timer.report()

    # Per-class precision / recall / F1
    per_class = {}
    for ci, cls in enumerate(classes):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == ci and p == ci)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != ci and p == ci)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == ci and p != ci)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) else 0.0)

        per_class[cls] = {
            "precision": round(precision, 4),
            "recall":    round(recall,    4),
            "f1":        round(f1,        4),
            "support":   sum(1 for t in y_true if t == ci),
        }
        print(f"  {cls:12s}  "
              f"P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}  "
              f"(n={per_class[cls]['support']})")

    # Confusion matrix
    matrix = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        matrix[t][p] += 1

    print("\n  Confusion Matrix  (rows=true, cols=pred)")
    header = "           " + "  ".join(f"{c[:8]:>8}" for c in classes)
    print(f"  {header}")
    for i, row in enumerate(matrix):
        row_str = "  ".join(f"{v:>8}" for v in row)
        print(f"  {classes[i][:10]:>10s}  {row_str}")

    return {
        "accuracy":         round(accuracy, 4),
        "n_videos":         n_total,
        "n_correct":        n_correct,
        "per_class":        per_class,
        "confusion_matrix": matrix.tolist(),
        "latency":          timer.stats(),
        "video_results":    video_results,
    }

print("✅ Stage A function defined.")

✅ Stage A function defined.


In [76]:
# ── Stage B: SGG YOLO-World Detection Quality & Latency ───────────────────────

def validate_sgg(sgg_model, test_videos: list):
    """
    Runs YOLO-World on sampled frames and measures:
      - Detection rate  (% frames with ≥1 detection)
      - Avg objects per frame
      - Avg confidence of detections
      - Nodes and edges generated per scene
      - Latency per frame
    """
    banner("STAGE B — SGG YOLO-World Detection Quality & Latency")

    if not YOLO_AVAILABLE or sgg_model is None:
        warn("YOLO-World not available — skipping SGG validation.")
        return {}

    if not test_videos:
        warn("No test videos — skipping SGG validation.")
        return {}

    timer_yolo = LatencyTimer("YOLO-World inference / frame")
    timer_sgg  = LatencyTimer("SGG graph build / frame")

    total_frames_checked    = 0
    frames_with_detections  = 0
    all_obj_counts          = []
    all_confidences         = []
    all_node_counts         = []
    all_edge_counts         = []
    video_sgg_results       = []

    for video_path, _ in test_videos:
        try:
            frames, _, _ = sample_frames(video_path)
        except IOError as e:
            fail(str(e)); continue

        video_nodes      = 0
        video_edges      = 0
        video_detections = 0

        for _, bgr in frames:
            total_frames_checked += 1

            results = timer_yolo.measure(
                lambda f=bgr: sgg_model.predict(f, verbose=False)
            )

            n_det = len(results[0].boxes) if results else 0
            all_obj_counts.append(n_det)

            if n_det > 0:
                frames_with_detections += 1
                confs = results[0].boxes.conf.cpu().numpy()
                all_confidences.extend(confs.tolist())
                video_detections += n_det

            def _build_graph(res=results):
                nodes, edges = [], []
                if not res or len(res[0].boxes) == 0:
                    return nodes, edges
                boxes   = res[0].boxes.xyxy.cpu().numpy()
                classes = res[0].boxes.cls.cpu().numpy()
                conf_v  = res[0].boxes.conf.cpu().numpy()
                names   = res[0].names
                for i, cid in enumerate(classes):
                    if conf_v[i] >= 0.2:
                        label = names[int(cid)].replace("_"," ").title()
                        nodes.append({"id": f"n{i}", "label": label})
                ids = list(range(len(nodes)))
                for i in ids:
                    for j in ids:
                        if i >= j: continue
                        iou = calculate_iou(boxes[i], boxes[j])
                        if iou > 0.05:
                            edges.append({"src": f"n{i}", "tgt": f"n{j}", "rel": "interacting_with"})
                        elif iou > 0.0:
                            edges.append({"src": f"n{i}", "tgt": f"n{j}", "rel": "adjacent_to"})
                return nodes, edges

            nodes, edges = timer_sgg.measure(_build_graph)
            all_node_counts.append(len(nodes))
            all_edge_counts.append(len(edges))
            video_nodes += len(nodes)
            video_edges += len(edges)

        info(f"{Path(video_path).name:35s}  "
             f"detections={video_detections:3d}  "
             f"nodes={video_nodes:3d}  "
             f"edges={video_edges:3d}")

        video_sgg_results.append({
            "video":       video_path,
            "detections":  video_detections,
            "total_nodes": video_nodes,
            "total_edges": video_edges,
        })

    detection_rate = frames_with_detections / total_frames_checked if total_frames_checked else 0
    avg_objs       = float(np.mean(all_obj_counts))  if all_obj_counts  else 0
    avg_conf       = float(np.mean(all_confidences)) if all_confidences else 0
    avg_nodes      = float(np.mean(all_node_counts)) if all_node_counts else 0
    avg_edges      = float(np.mean(all_edge_counts)) if all_edge_counts else 0

    banner("SGG — Results Summary")
    ok(f"Detection rate   : {frames_with_detections}/{total_frames_checked} frames "
       f"= {detection_rate*100:.1f}%")
    ok(f"Avg objects/frame: {avg_objs:.2f}")
    ok(f"Avg confidence   : {avg_conf:.3f}")
    ok(f"Avg nodes/frame  : {avg_nodes:.2f}")
    ok(f"Avg edges/frame  : {avg_edges:.2f}")
    timer_yolo.report()
    timer_sgg.report()

    return {
        "detection_rate":        round(detection_rate, 4),
        "frames_checked":        total_frames_checked,
        "avg_objects_per_frame": round(avg_objs,  4),
        "avg_confidence":        round(avg_conf,  4),
        "avg_nodes_per_frame":   round(avg_nodes, 4),
        "avg_edges_per_frame":   round(avg_edges, 4),
        "yolo_latency":          timer_yolo.stats(),
        "sgg_latency":           timer_sgg.stats(),
        "video_results":         video_sgg_results,
    }

print("✅ Stage B function defined.")

✅ Stage B function defined.


In [77]:
# ── Stage C: Full Pipeline End-to-End Latency ─────────────────────────────────

def validate_pipeline_latency(env_model, device, sgg_model, test_videos: list, classes: list):
    """
    Times the full pipeline per video (CNN + YOLO + graph build).
    This is the number the LLM team needs:
      how long does the feature extraction take before the LLM even runs?
    """
    banner("STAGE C — Full Pipeline End-to-End Latency")

    timer_full = LatencyTimer("Full pipeline / video")
    results    = []

    for video_path, true_label in test_videos:
        def _run_pipeline(vp=video_path):
            frames, fps_val, total = sample_frames(vp)
            accumulated = torch.zeros(len(classes)).to(device)
            best_res    = None
            max_det     = -1

            for _, bgr in frames:
                if sgg_model:
                    res = sgg_model.predict(bgr, verbose=False)
                    n   = len(res[0].boxes) if res else 0
                    if n > max_det:
                        max_det  = n
                        best_res = res
                rgb    = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                tensor = PREPROCESS(Image.fromarray(rgb)).unsqueeze(0).to(device)
                with torch.no_grad():
                    out   = env_model(tensor)
                    probs = torch.nn.functional.softmax(out[0], dim=0)
                    accumulated += probs

            avg  = accumulated / len(frames)
            conf, idx = torch.max(avg, 0)
            return classes[idx.item()], conf.item()

        predicted, confidence = timer_full.measure(_run_pipeline)
        correct = (predicted == true_label)
        symbol  = "✔" if correct else "✘"
        sc      = C.GREEN if correct else C.RED

        last_t = timer_full.times[-1]
        print(f"  {sc}{symbol}{C.END}  {Path(video_path).name:35s}  "
              f"pred={predicted:12s}  conf={confidence:.3f}  "
              f"time={last_t:.1f}ms")

        results.append({
            "video":      video_path,
            "true_label": true_label,
            "pred_label": predicted,
            "confidence": round(confidence, 4),
            "correct":    correct,
            "latency_ms": round(last_t, 2),
        })

    banner("Pipeline Latency Summary")
    timer_full.report()

    stats = timer_full.stats()
    if stats["mean_ms"] < 500:
        ok(f"Pipeline is FAST — avg {stats['mean_ms']:.0f}ms/video ✅")
    elif stats["mean_ms"] < 2000:
        warn(f"Pipeline is MODERATE — avg {stats['mean_ms']:.0f}ms/video — acceptable for batch use")
    else:
        warn(f"Pipeline is SLOW — avg {stats['mean_ms']:.0f}ms/video — consider optimising")

    return {"latency": stats, "video_results": results}

print("✅ Stage C function defined.")


✅ Stage C function defined.


In [78]:
def run_smoke_tests(env_model, device, sgg_model, classes: list):
    """
    Fast checks that don't require labelled data.
    Proves the models are loaded and runnable.
    """
    banner("STAGE D — Smoke Tests (model sanity checks)")

    passed = 0
    failed = 0

    # Test 1: CNN forward pass on a random tensor
    try:
        dummy = torch.randn(1, 3, 224, 224).to(device)
        with torch.no_grad():
            out = env_model(dummy)
        assert out.shape == (1, len(classes)), \
            f"Expected ({len(classes)},) output, got {out.shape}"
        ok(f"CNN forward pass → output shape {tuple(out.shape)} ✅")
        passed += 1
    except Exception as e:
        fail(f"CNN forward pass failed: {e}")
        failed += 1

    # Test 2: CNN output is valid probability distribution
    try:
        dummy = torch.randn(1, 3, 224, 224).to(device)
        with torch.no_grad():
            out   = env_model(dummy)
            probs = torch.softmax(out[0], dim=0)
        assert abs(probs.sum().item() - 1.0) < 1e-4, "Probabilities don't sum to 1"
        assert all(p >= 0 for p in probs.tolist()), "Negative probability"
        ok(f"CNN softmax sanity → probs sum to {probs.sum().item():.6f} ✅")
        passed += 1
    except Exception as e:
        fail(f"CNN probability check failed: {e}")
        failed += 1

    # Test 3: CNN inference time on a single frame
    try:
        dummy = torch.randn(1, 3, 224, 224).to(device)
        t0 = time.perf_counter()
        for _ in range(10):
            with torch.no_grad():
                env_model(dummy)
        avg_ms = (time.perf_counter() - t0) / 10 * 1000
        ok(f"CNN single-frame latency → {avg_ms:.1f}ms avg over 10 runs ✅")
        passed += 1
    except Exception as e:
        fail(f"CNN latency test failed: {e}")
        failed += 1

    # Test 4: YOLO on a blank frame
    if sgg_model:
        try:
            blank = np.zeros((480, 640, 3), dtype=np.uint8)
            t0    = time.perf_counter()
            res   = sgg_model.predict(blank, verbose=False)
            ms    = (time.perf_counter() - t0) * 1000
            n_det = len(res[0].boxes) if res else 0
            ok(f"YOLO blank frame → {n_det} detections in {ms:.1f}ms ✅")
            passed += 1
        except Exception as e:
            fail(f"YOLO blank frame failed: {e}")
            failed += 1

        # Test 5: YOLO on a random-noise frame
        try:
            noise = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
            res   = sgg_model.predict(noise, verbose=False)
            ok(f"YOLO noise frame → {len(res[0].boxes)} detections ✅")
            passed += 1
        except Exception as e:
            fail(f"YOLO noise frame failed: {e}")
            failed += 1
    else:
        warn("YOLO smoke tests skipped (model not loaded)")

    # Test 6: IOU function edge cases
    try:
        iou_overlap = calculate_iou([0,0,10,10], [5,5,15,15])
        iou_none    = calculate_iou([0,0,5,5],   [10,10,20,20])
        assert 0 < iou_overlap < 1, "Partial overlap IOU should be between 0 and 1"
        assert iou_none == 0.0, "Non-overlapping boxes should have IOU = 0"
        ok(f"IOU helper → partial={iou_overlap:.3f}, none={iou_none:.3f} ✅")
        passed += 1
    except Exception as e:
        fail(f"IOU test failed: {e}")
        failed += 1

    banner(f"Smoke Tests — {passed} passed, {failed} failed")
    if failed == 0:
        ok("All smoke tests passed — pipeline is ready for integration ✅")
    else:
        warn(f"{failed} test(s) failed — review before integration")

    return {"passed": passed, "failed": failed}

print("✅ Stage D function defined.")


✅ Stage D function defined.


In [79]:
def discover_test_videos(test_dir: str, classes: list) -> list:
    """
    Walk test_dir looking for  <test_dir>/<class_name>/*.mp4
    Returns list of  (video_path, label)
    """
    test_dir = Path(test_dir)
    results  = []
    for cls in classes:
        cls_dir = test_dir / cls
        if not cls_dir.exists():
            warn(f"No folder found for class '{cls}' at {cls_dir}")
            continue
        for ext in ("*.mp4", "*.avi", "*.mov", "*.mkv"):
            for vp in cls_dir.glob(ext):
                results.append((str(vp), cls))
    if results:
        ok(f"Discovered {len(results)} labelled test videos")
    else:
        warn("No labelled test videos discovered")
    return results


def save_report(report: dict, output_path: str = "validation_report.json"):
    with open(output_path, "w") as f:
        json.dump(report, f, indent=2)
    ok(f"Full report saved → {output_path}")

print("✅ Helper functions defined.")

✅ Helper functions defined.


In [80]:
banner("Collecting Test Videos")

test_videos = []

# Option A: labelled folder
if TEST_DIR:
    test_videos = discover_test_videos(TEST_DIR, CLASSES)

# Option B: single video
if SINGLE_VIDEO and SINGLE_LABEL:
    test_videos.append((SINGLE_VIDEO, SINGLE_LABEL))
    ok(f"Single video added: {SINGLE_VIDEO}  label={SINGLE_LABEL}")
elif SINGLE_VIDEO:
    warn("SINGLE_VIDEO provided without SINGLE_LABEL — accuracy tests skipped for it.")
    test_videos.append((SINGLE_VIDEO, CLASSES[0]))

if not test_videos:
    warn("No test videos found — smoke tests will still run.")
else:
    info(f"Total test videos: {len(test_videos)}")


════════════════════════════════════════════════════════════
════════════════════════════════════════════════════════════
  ⚠  No folder found for class 'classroom' at /content/drive/MyDrive/Colab Notebooks/Data/classroom
  ⚠  No folder found for class 'home' at /content/drive/MyDrive/Colab Notebooks/Data/home
  ✔  Discovered 73 labelled test videos
  ✔  Single video added: /content/drive/MyDrive/Colab Notebooks/Data/living_room/3769952-hd_1920_1080_25fps.mp4  label=home
  ℹ  Total test videos: 74


In [ ]:
# Load models after all functions are defined.
env_model, device = load_env_model(ENV_MODEL_PATH, num_classes=len(CLASSES))
sgg_model = None
if not NO_SGG:
    sgg_model = load_sgg_model(YOLO_WEIGHTS, DEFAULT_SGG_VOCAB)

# Existing code from bmC-GWJiJjNo
from datetime import datetime

banner("SGG PIPELINE VALIDATION — Starting")
print(f"  Model   : {ENV_MODEL_PATH}")
print(f"  Classes : {CLASSES}")
print(f"  Device  : {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"  Time    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

full_report = {
    "timestamp":  datetime.now().isoformat(),
    "model_path": ENV_MODEL_PATH,
    "classes":    CLASSES,
    "device":     "GPU" if torch.cuda.is_available() else "CPU",
}

# Stage D — Smoke tests (run first, no data required)
full_report["smoke_tests"] = run_smoke_tests(env_model, device, sgg_model, CLASSES)

# Stage A — CNN accuracy
full_report["env_accuracy"] = validate_env_model(env_model, device, test_videos, CLASSES)

# Stage B — SGG quality
full_report["sgg_quality"] = validate_sgg(sgg_model, test_videos)

# Stage C — Full pipeline latency
full_report["pipeline_latency"] = validate_pipeline_latency(
    env_model, device, sgg_model, test_videos, CLASSES
)


════════════════════════════════════════════════════════════
  SGG PIPELINE VALIDATION — Starting
════════════════════════════════════════════════════════════
  Model   : /content/drive/MyDrive/Other/places365_environment_model_new.pth
  Classes : ['classroom', 'home', 'office']
  Device  : CPU
  Time    : 2026-03-17 19:44:52

════════════════════════════════════════════════════════════
  STAGE D — Smoke Tests (model sanity checks)
════════════════════════════════════════════════════════════
  ✔  CNN forward pass → output shape (1, 3) ✅
  ✔  CNN softmax sanity → probs sum to 1.000000 ✅
  ✔  CNN single-frame latency → 132.5ms avg over 10 runs ✅
  ✔  YOLO blank frame → 0 detections in 968.3ms ✅
  ✔  YOLO noise frame → 0 detections ✅
  ✔  IOU helper → partial=0.143, none=0.000 ✅

════════════════════════════════════════════════════════════
  Smoke Tests — 6 passed, 0 failed
════════════════════════════════════════════════════════════
  ✔  All smoke tests passed — pipeline is ready for in

In [ ]:
banner("FINAL VALIDATION VERDICT")

smoke_ok  = full_report["smoke_tests"]["failed"] == 0
acc_val   = full_report["env_accuracy"].get("accuracy", None)
det_rate  = full_report["sgg_quality"].get("detection_rate", None)
lat_mean  = full_report["pipeline_latency"]["latency"].get("mean_ms", None)

if smoke_ok:
    ok("Smoke tests      : ALL PASSED ✅")
else:
    fail(f"Smoke tests      : {full_report['smoke_tests']['failed']} FAILED ❌")

if acc_val is not None:
    colour = C.GREEN if acc_val >= 0.70 else C.YELLOW if acc_val >= 0.50 else C.RED
    print(f"  {colour}{'✔' if acc_val >= 0.70 else '⚠'}  "
          f"Env Accuracy     : {acc_val*100:.1f}%  "
          f"{'(Good ✅)' if acc_val >= 0.70 else '(Needs improvement ⚠)'}{C.END}")

if det_rate is not None:
    colour = C.GREEN if det_rate >= 0.50 else C.YELLOW
    print(f"  {colour}✔  Detection rate  : {det_rate*100:.1f}%{C.END}")

if lat_mean is not None:
    colour = C.GREEN if lat_mean < 1000 else C.YELLOW
    print(f"  {colour}✔  Pipeline latency: {lat_mean:.0f}ms avg/video{C.END}")

In [ ]:
# Save JSON report to disk
save_report(full_report, REPORT_PATH)

banner("Validation Complete")
print(f"\n  Report saved to: {REPORT_PATH}")
